<a href="https://colab.research.google.com/github/AbdulrahmanB-25/Machine_Learning_Competition/blob/main/EDA_of_DataSets/EDA_Services.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [49]:
import pandas as pd
import geopandas as gpd
import ast
import json

# 1. Load the dataset from GitHub
url = 'https://raw.githubusercontent.com/AbdulrahmanB-25/Machine_Learning_Competition/main/DataSets/27k_Riyadh_Places.csv'
df_places = pd.read_csv(url)

# 2. Define our 14 Expanded Master Categories
NORM_MAP = {
    'dining_cafe': ['Coffee Shop', 'Café', 'Restaurant', 'Burger Joint', 'Bakery', 'Breakfast Spot', 'Juice Bar', 'Dessert Shop', 'Food Truck'],
    'med_facilities': ['Hospital', 'Medical Center', 'Clinic', 'Emergency Room', 'Dentist', 'Doctor'],
    'health_retail': ['Pharmacy', 'Drugstore', 'Optical Shop'],
    'fitness_care': ['Gym', 'Yoga Studio', 'Martial Arts', 'Spa', 'Salon', 'Barber', 'Cosmetics'],
    'edu_primary': ['Nursery', 'Kindergarten', 'Elementary School', 'Daycare'],
    'edu_higher': ['High School', 'University', 'College', 'Training Center', 'Language School', 'Library'],
    'religious': ['Mosque', 'Prayer Room'],
    'essential_retail': ['Market', 'Grocery', 'Supermarket', 'Laundry', 'Petrol Station', 'Hardware', 'ATM'],
    'parks_green': ['Park', 'Garden', 'Plaza', 'Promenade', 'Botanical Garden'],
    'sports_play': ['Soccer Field', 'Padel Court', 'Playground', 'Tennis Court', 'Basketball Court', 'Skate Park'],
    'pedestrian': ['Walking Trail', 'Pedestrian Street', 'Bike Trail'],
    'resort_rural_retreats': ['Farm', 'Campground', 'Stable', 'Resort', 'Theme Park'],
    'gov_civil': ['Police', 'Post Office', 'Courthouse', 'Embassy', 'Municipality', 'Fire Station'],
    'malls_shopping': ['Shopping Mall', 'Shopping Center', 'Department Store', 'Outlet Mall', 'Commercial Plaza']
}

# 3. Build reverse lookup for speed
reverse_map = {sc.lower(): k for k, v in NORM_MAP.items() for sc in v}

def normalize_category(raw):
    try: cats = ast.literal_eval(raw)
    except: return 'other'
    for cat in cats:
        for keyword, norm_key in reverse_map.items():
            if keyword in cat.lower(): return norm_key
    return 'other'

df_places['norm_category'] = df_places['category'].apply(normalize_category)
# --- PART 4: GEOSPATIAL PROCESSING (FIXED) ---
df_places["latitude"]  = pd.to_numeric(df_places["latitude"],  errors="coerce")
df_places["longitude"] = pd.to_numeric(df_places["longitude"], errors="coerce")
df_places = df_places.dropna(subset=["latitude", "longitude"]).reset_index(drop=True)

# Load Riyadh districts
geojson_url = "https://raw.githubusercontent.com/AbdulrahmanB-25/Machine_Learning_Competition/7d9bc540cd6df0e61dec3816c1d990ddb64edfa6/Saudi-Arabia-Regions-Cities-and-Districts-3.0.0/geojson/districts.geojson"
districts = gpd.read_file(geojson_url)
riyadh_districts = districts[districts["city_id"] == 3].copy().to_crs("EPSG:4326")

# Convert to GeoDataFrame
gdf = gpd.GeoDataFrame(
    df_places,
    geometry=gpd.points_from_xy(df_places["longitude"], df_places["latitude"]),
    crs="EPSG:4326"
)

# Spatial join (exact match)
joined = gpd.sjoin(gdf, riyadh_districts[["name_en", "geometry"]], how="left", predicate="within")
joined["neighborhoods"] = joined["name_en"]

# Fallback: Nearest Join
missing_mask = joined["neighborhoods"].isna()
if missing_mask.sum() > 0:
    print(f"Fallback: Tagging {missing_mask.sum()} points using nearest neighbor...")
    gdf_missing_proj = gdf[missing_mask].copy().to_crs(epsg=32638)
    riyadh_districts_proj = riyadh_districts.to_crs(epsg=32638)

    # 1. Perform the nearest join
    nearest = gpd.sjoin_nearest(gdf_missing_proj, riyadh_districts_proj[["name_en", "geometry"]], how="left")

    # 2. IMPORTANT FIX: Remove duplicates if a point has 2+ "nearest" neighbors
    nearest = nearest[~nearest.index.duplicated(keep='first')]

    # 3. Update the values
    joined.loc[missing_mask, "neighborhoods"] = nearest["name_en"].values

df_places["neighborhoods"] = joined["neighborhoods"].values

# --- PART 5: THE RESHAPE (REWRITE) ---
# Filter out 'other' and pivot
df_final_pivot = df_places[df_places['norm_category'] != 'other'].copy()

# Pivot to counts
neighborhood_profiles = df_final_pivot.groupby(['neighborhoods', 'norm_category']).size().unstack(fill_value=0)

# Ensure all 14 columns exist
neighborhood_profiles = neighborhood_profiles.reindex(columns=NORM_MAP.keys(), fill_value=0)

# Save result
neighborhood_profiles.to_csv("Riyadh_Neighborhood_Service_Profiles.csv")
print("Successfully saved to Riyadh_Neighborhood_Service_Profiles.csv")
neighborhood_profiles.head()

Fallback: Tagging 3611 points using nearest neighbor...
Successfully saved to Riyadh_Neighborhood_Service_Profiles.csv


norm_category,dining_cafe,med_facilities,health_retail,fitness_care,edu_primary,edu_higher,religious,essential_retail,parks_green,sports_play,pedestrian,resort_rural_retreats,gov_civil,malls_shopping
neighborhoods,,,,,,,,,,,,,,
2nd Industrial City,42,2,1,0,1,0,0,4,1,3,0,0,3,4
Al Amal Dist.,22,1,0,0,0,0,0,4,0,0,0,0,2,1
Al Andalus Dist.,59,4,3,5,0,0,2,4,1,1,0,0,0,1
Al Aqeeq Dist.,154,4,5,0,0,0,2,4,4,0,0,0,0,2
Al Arid Dist.,241,7,6,1,1,3,13,20,19,18,0,36,1,2


<class 'pandas.core.frame.DataFrame'>
Index: 166 entries, 2nd Industrial City to West Umm Al Hamam Dist.
Data columns (total 14 columns):
 #   Column                 Non-Null Count  Dtype
---  ------                 --------------  -----
 0   dining_cafe            166 non-null    int64
 1   med_facilities         166 non-null    int64
 2   health_retail          166 non-null    int64
 3   fitness_care           166 non-null    int64
 4   edu_primary            166 non-null    int64
 5   edu_higher             166 non-null    int64
 6   religious              166 non-null    int64
 7   essential_retail       166 non-null    int64
 8   parks_green            166 non-null    int64
 9   sports_play            166 non-null    int64
 10  pedestrian             166 non-null    int64
 11  resort_rural_retreats  166 non-null    int64
 12  gov_civil              166 non-null    int64
 13  malls_shopping         166 non-null    int64
dtypes: int64(14)
memory usage: 23.5+ KB
